# Lab 2: Anomaly Detection in Network Traffic

**Students:** Aviv Heller, Shaked Yakobi

This notebook builds an end-to-end anomaly detection pipeline to identify SSH brute-force attacks in synthetic network traffic data using Isolation Forest.

**MITRE ATT&CK Technique:** [T1110.001 - Brute Force: Password Guessing](https://attack.mitre.org/techniques/T1110/001/)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, confusion_matrix

sns.set_style("whitegrid")

## Part 1: Dataset Preparation

We generate synthetic network connection logs simulating a day of traffic. The dataset contains **97% normal traffic** and **3% SSH brute-force attack traffic** from a single attacker IP.

**Attack scenario (T1110.001):** An attacker at IP `10.0.0.200` launches a password guessing attack against the SSH service (port 22). The attack is characterized by rapid, short-lived connections with many failed login attempts, concentrated in a 2-hour window.

In [ ]:
np.random.seed(42)

n_total = 10000
attack_ratio = 0.03
n_attack = int(n_total * attack_ratio)
n_normal = n_total - n_attack

# --- Normal traffic ---
normal_ips = [f"192.168.1.{i}" for i in range(1, 51)]
all_ports = [80, 443, 8080, 53, 22, 3389, 25, 110, 993, 8443]

base_date = pd.Timestamp("2026-03-01")
normal_timestamps = base_date + pd.to_timedelta(
    np.random.uniform(0, 24 * 3600, n_normal), unit="s"
)

normal_data = pd.DataFrame({
    "timestamp": normal_timestamps,
    "src_ip": np.random.choice(normal_ips, n_normal),
    "dst_port": np.random.choice(all_ports, n_normal),
    "bytes_sent": np.clip(np.random.normal(500, 200, n_normal), 50, 5000).astype(int),
    "duration_ms": np.clip(np.random.normal(300, 150, n_normal), 10, 5000).round(1),
    "protocol": np.random.choice(["TCP", "UDP", "ICMP"], n_normal, p=[0.7, 0.2, 0.1]),
    "failed_logins": np.random.choice([0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 2], n_normal),
    "label": 0
})

# --- Attack traffic (SSH brute force from single IP) ---
attack_timestamps = base_date + pd.to_timedelta(
    np.random.uniform(2 * 3600, 4 * 3600, n_attack), unit="s"
)

attack_data = pd.DataFrame({
    "timestamp": attack_timestamps,
    "src_ip": "10.0.0.200",
    "dst_port": 22,
    "bytes_sent": np.clip(np.random.normal(120, 30, n_attack), 40, 250).astype(int),
    "duration_ms": np.clip(np.random.normal(50, 15, n_attack), 5, 150).round(1),
    "protocol": "TCP",
    "failed_logins": np.random.randint(3, 21, n_attack),
    "label": 1
})

# --- Combine and sort by time ---
df = pd.concat([normal_data, attack_data], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df = df.sort_values("timestamp").reset_index(drop=True)

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

In [ ]:
df.head(10)

## Part 2: Exploratory Data Analysis

We examine the dataset structure, distributions, and patterns to understand what "normal" traffic looks like before applying anomaly detection.

In [ ]:
print(f"Total records: {len(df)}")
print(f"Number of features: {df.shape[1]}")
print(f"\nClass distribution:")
print(df["label"].value_counts().rename({0: "Normal", 1: "Attack"}))
print(f"\nAttack percentage: {df['label'].mean() * 100:.1f}%")
print(f"\nDescriptive statistics:")
df.describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col in zip(axes, ["bytes_sent", "duration_ms", "failed_logins"]):
    df[df["label"] == 0][col].hist(ax=ax, bins=40, alpha=0.6, label="Normal", color="steelblue")
    df[df["label"] == 1][col].hist(ax=ax, bins=40, alpha=0.6, label="Attack", color="crimson")
    ax.set_title(f"Distribution of {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Count")
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot of destination ports by class
plot_df = df.copy()
plot_df["class"] = plot_df["label"].map({0: "Normal", 1: "Attack"})

sns.countplot(data=plot_df, x="dst_port", hue="class", ax=axes[0],
              palette={"Normal": "steelblue", "Attack": "crimson"})
axes[0].set_title("Connection Count by Destination Port")
axes[0].set_xlabel("Destination Port")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=45)

# Time distribution (hour of day)
plot_df["hour"] = plot_df["timestamp"].dt.hour
sns.histplot(data=plot_df, x="hour", hue="class", bins=24, ax=axes[1],
             palette={"Normal": "steelblue", "Attack": "crimson"}, multiple="stack")
axes[1].set_title("Connection Count by Hour of Day")
axes[1].set_xlabel("Hour")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

### Analytical Summary

Normal network traffic is distributed uniformly across the 24-hour period, with connections spread across all 50 source IPs and 10 destination ports. The majority of normal connections have moderate byte volumes (mean ~500 bytes) and connection durations (mean ~300ms), with failed login counts overwhelmingly at zero. The attack traffic, representing 3% of records, deviates sharply: it originates from a single IP (10.0.0.200), targets exclusively port 22 (SSH), occurs in a concentrated 2-hour burst between 02:00 and 04:00, and exhibits high failed login counts (3-20 per session) with smaller packet sizes and shorter durations. These patterns across multiple features make the brute-force activity clearly distinguishable from the normal baseline.

## Part 3: Anomaly Detection with Isolation Forest

Isolation Forest works by randomly selecting a feature and a split value to isolate observations. Anomalies, being rare and different, require fewer random splits to be isolated — resulting in shorter average path lengths in the ensemble of trees.

In [ ]:
# Extract hour as numeric feature from timestamp
df["hour"] = df["timestamp"].dt.hour + df["timestamp"].dt.minute / 60

# Encode categorical features
le_ip = LabelEncoder()
le_proto = LabelEncoder()

df_model = df.copy()
df_model["src_ip_enc"] = le_ip.fit_transform(df_model["src_ip"])
df_model["protocol_enc"] = le_proto.fit_transform(df_model["protocol"])

# Select and scale features
feature_names = ["src_ip_enc", "dst_port", "bytes_sent", "duration_ms",
                 "protocol_enc", "failed_logins", "hour"]

scaler = StandardScaler()
X = scaler.fit_transform(df_model[feature_names])

print(f"Feature matrix shape: {X.shape}")
print(f"Features: {feature_names}")

In [ ]:
iso_forest = IsolationForest(
    n_estimators=200,
    contamination=0.03,
    max_samples="auto",
    random_state=42
)

iso_forest.fit(X)

# Get anomaly scores and predictions
df["anomaly_score"] = iso_forest.decision_function(X)
df["anomaly_label"] = iso_forest.predict(X)

# Remap: sklearn uses 1=normal, -1=anomaly -> 0=normal, 1=anomaly
df["anomaly_label"] = (df["anomaly_label"] == -1).astype(int)

print(f"Anomalies detected: {df['anomaly_label'].sum()}")
print(f"Normal detected: {(df['anomaly_label'] == 0).sum()}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Anomaly score histogram
axes[0].hist(df[df["label"] == 0]["anomaly_score"], bins=50, alpha=0.6,
             label="Normal (ground truth)", color="steelblue")
axes[0].hist(df[df["label"] == 1]["anomaly_score"], bins=50, alpha=0.6,
             label="Attack (ground truth)", color="crimson")
axes[0].set_title("Distribution of Anomaly Scores")
axes[0].set_xlabel("Anomaly Score (lower = more anomalous)")
axes[0].set_ylabel("Count")
axes[0].axvline(x=0, color="black", linestyle="--", alpha=0.5, label="Decision boundary")
axes[0].legend()

# Detection counts bar chart
counts = df["anomaly_label"].value_counts().sort_index()
bars = axes[1].bar(["Normal", "Anomaly"], [counts.get(0, 0), counts.get(1, 0)],
                    color=["steelblue", "crimson"])
axes[1].set_title("Detection Counts")
axes[1].set_ylabel("Count")
for bar in bars:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 str(int(bar.get_height())), ha="center", va="bottom", fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
print("Classification Report (Ground Truth vs. Isolation Forest):\n")
print(classification_report(
    df["label"], df["anomaly_label"],
    target_names=["Normal", "Attack"]
))

# Confusion matrix
cm = confusion_matrix(df["label"], df["anomaly_label"])
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Predicted Normal", "Predicted Anomaly"],
            yticklabels=["Actual Normal", "Actual Attack"])
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

## Part 4: 2D Visualization with PCA

We reduce the 7-dimensional feature space to 2 principal components to visualize how the model separates normal traffic from anomalies.

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Model predictions
axes[0].scatter(X_pca[df["anomaly_label"] == 0, 0], X_pca[df["anomaly_label"] == 0, 1],
                c="steelblue", alpha=0.3, s=10, label="Normal (predicted)")
axes[0].scatter(X_pca[df["anomaly_label"] == 1, 0], X_pca[df["anomaly_label"] == 1, 1],
                c="crimson", alpha=0.7, s=20, label="Anomaly (predicted)")
axes[0].set_title("PCA Projection - Isolation Forest Predictions")
axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
axes[0].legend()

# Plot 2: Ground truth
axes[1].scatter(X_pca[df["label"] == 0, 0], X_pca[df["label"] == 0, 1],
                c="steelblue", alpha=0.3, s=10, label="Normal (actual)")
axes[1].scatter(X_pca[df["label"] == 1, 0], X_pca[df["label"] == 1, 1],
                c="crimson", alpha=0.7, s=20, label="Attack (actual)")
axes[1].set_title("PCA Projection - Ground Truth Labels")
axes[1].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
axes[1].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
axes[1].legend()

plt.tight_layout()
plt.show()

### Interpretation

The PCA scatter plots show that the attack records form a distinct cluster, clearly separated from the main body of normal traffic. This separation is primarily driven by the combination of high failed login counts, exclusive targeting of port 22, and the unique source IP. The side-by-side comparison between the model's predictions and the ground truth labels shows strong agreement, confirming that the Isolation Forest successfully identified the anomalous region in the feature space.

## Conclusion

The Isolation Forest model successfully detected the SSH brute-force attack traffic (MITRE ATT&CK T1110.001) with high precision and recall. The synthetic attack pattern — a single source IP generating rapid, short-lived connections to port 22 with many failed login attempts — was clearly separable from the normal traffic baseline across multiple feature dimensions.

The PCA visualization confirmed that attack records form a distinct cluster in the reduced feature space, and the confusion matrix shows the model achieved strong detection performance. In a production SOC environment, this type of anomaly detection pipeline could be used to automatically flag suspicious activity and trigger alerts for security analysts.